# 30.04. Вычислительная проверка двуслойной модели

## Аннотация

Ноутбук содержит проверки передаточного импеданса плоской двуслойной среды: сравнение с опорными значениями, проверку аналитических производных, законов масштаба и взаимности. Все примеры синтетические. Они нужны, чтобы отделить ошибки реализации от ограничений самой модели при последующей обработке боковых записей.

**Текущее состояние.** В исходном ноутбуке нет сохранённых выходов вычислительных ячеек. При переоформлении расчёты не запускались. Поэтому эта редакция описывает проверяемые условия и критерии; прохождение проверок в ней ещё не зафиксировано.

Основной вопрос: согласуются ли разные способы вычисления одной и той же модельной величины? Даже положительный ответ устанавливает только внутреннюю математическую согласованность. Применимость плоской модели к грудной клетке проверяется отдельно по индивидуальной анатомии и расчётам методом конечных элементов.


## 1. Модель, входы и область проверки

Рассматривается вещественный передаточный импеданс четырёх точечных электродов на поверхности плоской двуслойной среды. Электроды лежат на одной прямой: токовые имеют координаты $+a$ и $-a$, потенциальные — $+b$ и $-b$. Приняты условия $0<b<a$ и размер сборки $L=2a$. Отношение $\beta=b/a$ задаёт её форму. Полярность потенциальной пары сохраняется во всех сравнениях, поскольку знак импеданса входит в результат.

Первый слой имеет толщину $h$ и эффективное удельное сопротивление $\rho_1$; второй слой полубесконечен и имеет сопротивление $\rho_2$. Оба слоя однородны и изотропны. Все длины в вычислительном ядре выражены в метрах, удельные сопротивления — в Ом·м, импеданс $Z$ — в омах. Толщина $h$ постоянна внутри каждого вычисления. В этом ноутбуке она задана как параметр синтетической среды и не является оценкой анатомической толщины добровольца.

Вывод формулы приведён в [30.01 — прямой двуслойной модели](30.01_Прямая_двуслойная_модель_боковой_сборки.md). Формула и производные реализованы в [вычислительном ядре](two_layer_model.py). Число членов ряда увеличивается до выполнения критериев сходимости, заданных в ядре. Здесь проверяется использование этой реализации; формула не переписывается в независимую копию.

Контактное сопротивление, конечная площадь электродов, кривизна поверхности, рёбра, неоднородность тканей и приборный тракт в данной постановке отсутствуют. Следовательно, возможное прохождение проверок не определяет допустимый размер реальной сборки.


In [ ]:
from pathlib import Path
import sys

import numpy as np

candidates = [Path.cwd(), Path.cwd() / "Colab Notebooks", Path.cwd().parent]
notebook_root = next((p for p in candidates if (p / "two_layer_model.py").exists()), None)
if notebook_root is None:
    raise FileNotFoundError("two_layer_model.py not found; run from the repository or Colab Notebooks directory")
sys.path.insert(0, str(notebook_root))

from two_layer_model import (
    apparent_resistivity,
    evaluate,
    geometry_from_size,
    transfer_impedance,
    transfer_impedance_coordinates,
)

## 2. Сравнение с опорными значениями

Первая проверка отвечает на вопрос, воспроизводит ли ядро выбранные значения кажущегося удельного сопротивления из таблицы документа 30.01. Входами служат $\rho_1=5$ Ом·м, $h=0{,}020$ м, три размера сборки и два значения $\rho_2$, явно перечисленные в вычислительной ячейке. Это параметры математического примера; экспериментальные записи не используются.

Ядро вычисляет знаковый импеданс и пересчитывает его в кажущееся удельное сопротивление $\rho_a$. Последнее обозначает сопротивление однородной среды, дающей тот же импеданс при том же монтаже. Оно не совпадает по определению с сопротивлением одного из слоёв.

Критерий сравнения — абсолютное расхождение $\rho_a$ не более $5\cdot10^{-5}$ Ом·м при нулевом относительном допуске. Это численный критерий для округлённых опорных значений. Прохождение сравнения не является независимой физической проверкой: таблица и ядро относятся к одной модели.


In [ ]:
rho1, h = 5.0, 0.020
expected = {
    (0.050, 15.0): 5.58577,
    (0.050, 25.0): 5.81610,
    (0.090, 15.0): 6.78717,
    (0.090, 25.0): 7.54596,
    (0.140, 15.0): 8.26561,
    (0.140, 25.0): 9.79495,
}

rows = []
for (size, rho2), target in expected.items():
    a, b = geometry_from_size(size)
    z = transfer_impedance(rho1, rho2, h, a, b)
    rho_a = apparent_resistivity(z, a, b)
    assert np.isclose(rho_a, target, rtol=0.0, atol=5e-5)
    rows.append((size * 1000, rho2, z, rho_a))

print("L, мм | rho2, Ом·м | Z, Ом | rho_a, Ом·м")
for row in rows:
    print("%5.0f | %11.1f | %5.2f | %11.5f" % row)

## 3. Производные и законы масштаба

Совпадение отдельных значений не исключает ошибку производных. Поэтому следующий расчёт сопоставляет аналитические производные $Z$ с центральными конечными разностями по $\rho_1$, $\rho_2$, $h$, $a$ и $b$. Производная по сопротивлению имеет единицу 1/м, по длине — Ом/м. Относительный шаг изменения каждого параметра равен $10^{-6}$. В коде заданы относительный допуск $2\cdot10^{-7}$ и абсолютный допуск $10^{-8}$ в единицах соответствующей производной. Допуски служат вычислительной проверке в выбранной точке; их универсальность здесь не исследуется.

Дополнительно проверяются две независимые от выбора единиц связи:

$$
\frac{\rho_1 Z_{\rho_1}+\rho_2 Z_{\rho_2}}{Z}=1, \tag{1}
$$

где $Z_p=\partial Z/\partial p$ — частная производная по параметру $p$ при фиксированных остальных параметрах. Равенство (1) выражает линейное изменение импеданса при умножении обоих удельных сопротивлений на один коэффициент.

$$
\frac{aZ_a+bZ_b+hZ_h}{Z}=-1, \tag{2}
$$

где $a$, $b$ и $h$ выражены в метрах, а $Z_a$, $Z_b$ и $Z_h$ — соответствующие производные в Ом/м. Равенство (2) выражает обратную пропорциональность импеданса общему масштабу длин при неизменных сопротивлениях. Обе левые части безразмерны; отношения определены при $Z\ne0$.

Сравнения для (1) и (2) используют абсолютный допуск $10^{-10}$ и относительный допуск NumPy по умолчанию. Поэтому абсолютный допуск сам по себе не задаёт строгость этих двух проверок. Это ограничение текущего кода следует учесть при отдельной ревизии вычислительных критериев.


In [ ]:
rho1, rho2, h = 5.0, 20.0, 0.020
a, b = geometry_from_size(0.140)
result = evaluate(rho1, rho2, h, a, b)
parameters = [rho1, rho2, h, a, b]
analytic = [result.d_rho1, result.d_rho2, result.d_h, result.d_a, result.d_b]
names = ["rho1", "rho2", "h", "a", "b"]

for index, (name, derivative) in enumerate(zip(names, analytic)):
    step = 1e-6 * parameters[index]
    plus, minus = parameters.copy(), parameters.copy()
    plus[index] += step
    minus[index] -= step
    numeric = (transfer_impedance(*plus) - transfer_impedance(*minus)) / (2 * step)
    assert np.isclose(derivative, numeric, rtol=2e-7, atol=1e-8)
    print(name, derivative, numeric)

resistivity_identity = (rho1 * result.d_rho1 + rho2 * result.d_rho2) / result.z
geometry_identity = (a * result.d_a + b * result.d_b + h * result.d_h) / result.z
assert np.isclose(resistivity_identity, 1.0, atol=1e-10)
assert np.isclose(geometry_identity, -1.0, atol=1e-10)
print("terms:", result.n_terms)

## 4. Взаимность и допустимая геометрия

Проверка взаимности устанавливает, сохраняется ли передаточный импеданс при обмене токовой и потенциальной пар на фиксированных физических координатах. Для этого используется общая координатная формула. Её результат сравнивается с формулой вложенного симметричного монтажа при абсолютном допуске $10^{-10}$ Ом и нулевом относительном допуске.

Подстановка $b>a$ в частную формулу не описывает такой обмен: она нарушает область определения принятого параметрического монтажа. Последняя часть ячейки проверяет, что недопустимая геометрия вызывает исключение. Таким образом, численная взаимность и проверка входных ограничений имеют разные критерии.


In [ ]:
direct = transfer_impedance_coordinates(a, -a, b, -b, rho1, rho2, h)
reciprocal = transfer_impedance_coordinates(b, -b, a, -a, rho1, rho2, h)
closed = transfer_impedance(rho1, rho2, h, a, b)
assert np.isclose(direct, reciprocal, rtol=0.0, atol=1e-10)
assert np.isclose(direct, closed, rtol=0.0, atol=1e-10)

try:
    transfer_impedance(rho1, rho2, h, b, a)
except ValueError:
    pass
else:
    raise AssertionError("invalid geometry b >= a must be rejected")

## 5. Выводы и следующий расчёт

В ноутбуке сформулированы три группы проверок: опорные значения, производные с законами масштаба и взаимность с контролем геометрии. Сохранённые результаты отсутствуют, поэтому ни одна группа в этой редакции не объявляется пройденной. Проверка однородного предела в эти ячейки не включена; её наличие в отдельном наборе тестов следует отличать от исполнения самого ноутбука.

Даже после прохождения всех проверок останется открытым вопрос, как геометрический масштаб меняет информативность сборки. Его рассматривает [30.05 — законы масштаба](30.05_Масштабные_законы_двуслойной_модели.ipynb). Следующий расчёт должен сохранять безразмерные отношения при изменении масштаба и отдельно проверять режим слабой чувствительности к глубокому слою. Затем серия 31 оценивает различимость параметров, а индивидуальные КТ/FEM-исследования серии 20 ограничивают применимость плоской модели.

Код, параметры, сохранённые выходы и счётчики исполнения при настоящем переоформлении сохранены. Историческая реализация остаётся в архиве; владельцем действующей формулы является [вычислительное ядро](two_layer_model.py).
